# PROJECT: Bank GoodCredit - Credit Score Prediction
# Client: Bank GoodCredit | Category: Banking - Risk
# Project Ref: PM-PR-0015
# Target Variable: Bad_label (0 = Good, 1 = Bad Credit History)
# Benchmark Gini: 37.9


# 1: IMPORTS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap

# 2: DATABASE CONNECTION & DATA LOADING

In [8]:
from urllib.parse import quote_plus

def load_data():
    """
    Connect to MySQL database and load all three tables:
      - Cust_Account    : Payment history, account details
      - Cust_Enquiry    : Loan/credit enquiry records
      - Cust_Demographics: Anonymised demographic features + target label
    """
    DB_CONFIG = {
        "host":     "18.136.157.135",
        "port":     3306,
        "user":     "dm_team1",
        "password": "DM!$Team&279@20!",
        "database": "project_banking"
    }

    # URL-encode the password to handle special characters like '@'
    encoded_password = quote_plus(DB_CONFIG['password'])

    engine = create_engine(
        f"mysql+pymysql://{DB_CONFIG['user']}:{encoded_password}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    )

    print("Loading tables from database...")
    df_account     = pd.read_sql("SELECT * FROM Cust_Account",     engine)
    df_enquiry     = pd.read_sql("SELECT * FROM Cust_Enquiry",     engine)
    df_demographics = pd.read_sql("SELECT * FROM Cust_Demographics", engine)

    print(f"  Cust_Account     : {df_account.shape}")
    print(f"  Cust_Enquiry     : {df_enquiry.shape}")
    print(f"  Cust_Demographics: {df_demographics.shape}")

    return df_account, df_enquiry, df_demographics

# 3: EXPLORATORY DATA ANALYSIS (EDA)

In [4]:
def perform_eda(df_account, df_enquiry, df_demographics):
    """
    EDA covers:
      - Shape, dtypes, missing values
      - Target class distribution
      - Distribution of key numeric columns
      - Correlation heatmap
    """
    print("\n" + "="*60)
    print("EXPLORATORY DATA ANALYSIS")
    print("="*60)

# 3.1 Basic Info

In [9]:
!pip install pymysql
df_account, df_enquiry, df_demographics = load_data()

for name, df in [("Account", df_account), ("Enquiry", df_enquiry), ("Demographics", df_demographics)]:
    print(f"\n[{name} Table]")
    print(f"  Shape       : {df.shape}")
    print(f"  Null counts :\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Loading tables from database...
  Cust_Account     : (186329, 21)
  Cust_Enquiry     : (413188, 6)
  Cust_Demographics: (23896, 83)

[Account Table]
  Shape       : (186329, 21)
  Null counts :
Series([], dtype: int64)

[Enquiry Table]
  Shape       : (413188, 6)
  Null counts :
Series([], dtype: int64)

[Demographics Table]
  Shape       : (23896, 83)
  Null counts :
Series([], dtype: int64)


# 3.2 Target Distribution

In [11]:
print("\n[Target Variable: Bad_label]")
print(df_demographics['Bad_label'].value_counts(normalize=True).round(4))

fig, ax = plt.subplots(figsize=(5, 4))
df_demographics['Bad_label'].value_counts().plot(
    kind='bar', ax=ax, color=['steelblue', 'tomato']
)
ax.set_title("Target Class Distribution")
ax.set_xticklabels(['Good (0)', 'Bad (1)'], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("eda_target_distribution.png", dpi=150)
plt.close()


[Target Variable: Bad_label]
Bad_label
0    0.958
1    0.042
Name: proportion, dtype: float64


# 3.3 Numeric Distribution (Account Table)

In [13]:
# The error "hist method requires numerical or datetime columns, nothing to plot."
# indicates that after df_account.select_dtypes(include=[np.number]),
# no columns were identified as numerical. This typically happens when
# columns that appear to be numbers or dates are actually stored as 'object' (string) type.

# 1. Convert date columns to datetime objects.
# Based on the data in 'df_account' snippet, 'dt_opened' and 'upload_dt' are likely date columns.
date_cols = ['dt_opened', 'upload_dt']
for col in date_cols:
    if col in df_account.columns:
        # Use dayfirst=True for date formats like '10-Nov-15' (day-month-year)
        df_account[col] = pd.to_datetime(df_account[col], errors='coerce', dayfirst=True)

# 2. Convert other 'object' type columns that are expected to be numeric.
# This loop iterates through all columns and attempts to convert 'object' columns to numeric.
# 'errors='coerce'' will turn any non-convertible values into NaN.
for col in df_account.columns:
    if df_account[col].dtype == 'object':
        df_account[col] = pd.to_numeric(df_account[col], errors='coerce')

# 3. Select columns that are now numerical (np.number) or datetime types.
# These are the types that the .hist() method can plot.
plottable_cols = df_account.select_dtypes(include=[np.number, 'datetime']).columns

# The original code attempts to plot the first 6 of these.
# If there are fewer than 6, it will plot all available.
numeric_and_datetime_cols_for_plot = plottable_cols[:6]

# 4. Check if there are any columns to plot before calling .hist()
if len(numeric_and_datetime_cols_for_plot) == 0:
    print("Warning: No numeric or datetime columns found in df_account for plotting histograms after type conversion.")
else:
    # Plot histograms for the selected columns
    df_account[numeric_and_datetime_cols_for_plot].hist(bins=30, figsize=(14, 8), color='steelblue', edgecolor='white')
    plt.suptitle("Numeric Feature Distributions - Account Table")
    plt.tight_layout()
    plt.savefig("eda_account_distributions.png", dpi=150)
    plt.close()

# 3.4 Correlation Heatmap (Demographics)

In [16]:
# Convert date columns in df_demographics to datetime objects
# 'dt_opened' and 'entry_time' are likely date columns
date_cols_demographics = ['dt_opened', 'entry_time']
for col in date_cols_demographics:
    if col in df_demographics.columns:
        df_demographics[col] = pd.to_datetime(df_demographics[col], errors='coerce', dayfirst=True)

# Convert other 'object' type columns in df_demographics that are expected to be numeric
for col in df_demographics.columns:
    if df_demographics[col].dtype == 'object':
        df_demographics[col] = pd.to_numeric(df_demographics[col], errors='coerce')

# Now, select numerical columns for correlation heatmap
num_demo = df_demographics.select_dtypes(include=[np.number])

# Check if num_demo is still empty after conversion
if num_demo.empty:
    print("Warning: No numerical columns found in df_demographics for correlation heatmap after type conversion.")
else:
    corr = num_demo.corr()
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3, annot=False)
    plt.title("Correlation Heatmap - Demographics Features")
    plt.tight_layout()
    plt.savefig("eda_correlation_heatmap.png", dpi=150)
    plt.close()

print("\nEDA plots saved: eda_target_distribution.png, eda_account_distributions.png, eda_correlation_heatmap.png")
# The return True statement should be outside the EDA function if this cell is meant to be part of a larger script.
# For a standalone cell, 'return True' at the top level is generally not necessary or standard.
# If this cell is part of a function (e.g., perform_eda), it would be valid there. For now, commenting it out.
# return True


EDA plots saved: eda_target_distribution.png, eda_account_distributions.png, eda_correlation_heatmap.png


##  4: FEATURE ENGINEERING

In [41]:
def engineer_features(df_account, df_enquiry, df_demographics):
    """
    Create features from account and enquiry tables, then merge with demographics.
    Features are based on the feature suggestions from the project brief.
    """
    print("\n" + "="*60)
    print("FEATURE ENGINEERING")
    print("="*60)

    # 4.1 Date Columns
    date_cols_acct = ['last_paymt_dt', 'opened_dt', 'closed_dt', 'reporting_dt',
                      'paymt_str_dt', 'paymt_end_dt', 'dt_opened']
    for col in date_cols_acct:
        if col in df_account.columns:
            df_account[col] = pd.to_datetime(df_account[col], errors='coerce')

    if 'enquiry_dt' in df_enquiry.columns:
        df_enquiry['enquiry_dt']  = pd.to_datetime(df_enquiry['enquiry_dt'],  errors='coerce')
    if 'dt_opened' in df_enquiry.columns:
        df_enquiry['dt_opened']   = pd.to_datetime(df_enquiry['dt_opened'],   errors='coerce')

    snapshot_date = df_account['reporting_dt'].max()
    print(f"  Snapshot date: {snapshot_date}")

    # Explicitly convert critical numeric columns in df_account to ensure they are numerical
    numeric_cols_to_convert_account = ['creditlimit', 'cur_balance_amt', 'cashlimit', 'amt_past_due', 'rateofinterest']
    for col in numeric_cols_to_convert_account:
        if col in df_account.columns:
            df_account[col] = pd.to_numeric(df_account[col], errors='coerce')

    # 4.2 Account-Level Features
    # F1: Duration between last payment and account open (per account)
    df_account['diff_lastpaymt_opened_dt'] = (
        df_account['last_paymt_dt'] - df_account['opened_dt']
    ).dt.days

    # F2: Payment history average DPD bucket 0-29
    def mean_dpd_0_29(history_str):
        """Return mean count of months with 0-29 DPD"""
        if pd.isnull(history_str):
            return np.nan
        counts = [1 for ch in str(history_str) if ch in ('0', '1', '2')]
        return np.mean(counts) if counts else 0

    df_account['dpd_0_29_ph1'] = df_account['paymenthistory1'].apply(mean_dpd_0_29)
    df_account['dpd_0_29_ph2'] = df_account['paymenthistory2'].apply(mean_dpd_0_29)
    df_account['payment_history_avg_dpd_0_29'] = df_account[['dpd_0_29_ph1', 'dpd_0_29_ph2']].mean(axis=1)

    # F3: Min months before 30+ DPD first appeared
    def min_months_to_30plus(history_str):
        if pd.isnull(history_str):
            return np.nan
        for i, ch in enumerate(str(history_str)):
            if ch not in ('0', '1', '2', 'X', 'x'):
                return i
        return len(str(history_str))  # never hit 30+

    df_account['min_months_last_30_plus_ph1'] = df_account['paymenthistory1'].apply(min_months_to_30plus)
    df_account['min_months_last_30_plus_ph2'] = df_account['paymenthistory2'].apply(min_months_to_30plus)

    # F4: Payment history length
    df_account['ph1_length'] = df_account['paymenthistory1'].apply(
        lambda x: len(str(x)) if pd.notnull(x) else np.nan
    )
    df_account['ph2_length'] = df_account['paymenthistory2'].apply(
        lambda x: len(str(x)) if pd.notnull(x) else np.nan
    )

    # F5: Utilisation = cur_balance / credit_limit
    df_account['utilisation'] = np.where(
        df_account['creditlimit'] > 0,
        df_account['cur_balance_amt'] / df_account['creditlimit'],
        np.nan
    )

    # 4.3 Aggregate to Customer Level
    agg_account = df_account.groupby('customer_no').agg(
        total_diff_lastpaymt_opened_dt   = ('diff_lastpaymt_opened_dt', 'sum'),
        mean_diff_lastpaymt_opened_dt    = ('diff_lastpaymt_opened_dt', 'mean'),
        payment_history_avg_dpd_0_29_bucket = ('payment_history_avg_dpd_0_29', 'mean'),
        min_months_last_30_plus          = ('min_months_last_30_plus_ph1', 'min'),
        payment_history_mean_length      = ('ph1_length', 'mean'),
        Ratio_currbalance_creditlimit    = ('utilisation', 'mean'),
        total_cur_balance                = ('cur_balance_amt', 'sum'),
        total_credit_limit               = ('creditlimit', 'sum'),
        total_cash_limit                 = ('cashlimit', 'sum'),
        mean_credit_limit                = ('creditlimit', 'mean'),
        mean_cash_limit                  = ('cashlimit', 'mean'),
        mean_cur_balance                 = ('cur_balance_amt', 'mean'),
        count_accounts                   = ('acct_type', 'count'),
        amt_past_due_total               = ('amt_past_due', 'sum'),
        mean_rate_of_interest            = ('rateofinterest', 'mean')
    ).reset_index()

    # F6: Utilisation trend
    agg_account['utilisation_trend'] = np.where(
        (agg_account['mean_credit_limit'] + agg_account['mean_cash_limit']) > 0,
        (agg_account['total_cur_balance'] / agg_account['total_credit_limit'].replace(0, np.nan)) /
        (agg_account['mean_cur_balance']  / (agg_account['mean_credit_limit'] + agg_account['mean_cash_limit'])),
        np.nan
    )

    print(f"  Account features aggregated: {agg_account.shape}")

    # 4.4 Enquiry Features
    # Ensure 'enq_amt' is numeric before calculations
    df_enquiry['enq_amt'] = pd.to_numeric(df_enquiry['enq_amt'], errors='coerce')

    df_enquiry['days_since_enquiry'] = (snapshot_date - df_enquiry['enquiry_dt']).dt.days

    agg_enquiry = df_enquiry.groupby('customer_no').agg(
        count_enquiry_recency_365 = ('days_since_enquiry', lambda x: (x <= 365).sum()),
        count_enquiry_recency_90  = ('days_since_enquiry', lambda x: (x <= 90).sum()),
        max_freq_enquiry          = ('enq_purpose', lambda x: x.mode()[0] if len(x) > 0 else np.nan),
        mean_diff_open_enquiry_dt = ('enquiry_dt', lambda x: (
            df_enquiry.loc[x.index, 'enquiry_dt'] -
            df_enquiry.loc[x.index, 'dt_opened']
        ).dt.days.mean()),
        perc_unsecured_others      = ('enq_purpose', lambda x: (x == 'Unsecured').sum() / len(x) if len(x) > 0 else 0),
        total_enquiry_amt         = ('enq_amt', 'sum'),
        mean_enquiry_amt          = ('enq_amt', 'mean'),
        total_enquiries           = ('enq_purpose', 'count'),
    ).reset_index()

    # Encode max_freq_enquiry
    le = LabelEncoder()
    agg_enquiry['max_freq_enquiry'] = le.fit_transform(
        agg_enquiry['max_freq_enquiry'].astype(str)
    )

    print(f"  Enquiry features aggregated: {agg_enquiry.shape}")

    # 4.5 Merge All Tables
    # Ensure 'customer_no' columns have consistent data types before merging
    df_demographics['customer_no'] = df_demographics['customer_no'].astype(str).str.replace('.0', '', regex=False).astype(int, errors='ignore')
    agg_account['customer_no'] = agg_account['customer_no'].astype(str).str.replace('.0', '', regex=False).astype(int, errors='ignore')
    agg_enquiry['customer_no'] = agg_enquiry['customer_no'].astype(str).str.replace('.0', '', regex=False).astype(int, errors='ignore')

    # Now proceed with merging
    df_final = df_demographics.merge(agg_account, on='customer_no', how='left')
    df_final = df_final.merge(agg_enquiry,  on='customer_no', how='left')

    print(f"  Final merged dataset: {df_final.shape}")
    return df_final

# 4.1 Date Columns

In [18]:
    date_cols_acct = ['last_paymt_dt', 'opened_dt', 'closed_dt', 'reporting_dt',
                      'paymt_str_dt', 'paymt_end_dt', 'dt_opened']
    for col in date_cols_acct:
        if col in df_account.columns:
            df_account[col] = pd.to_datetime(df_account[col], errors='coerce')

    if 'enquiry_dt' in df_enquiry.columns:
        df_enquiry['enquiry_dt']  = pd.to_datetime(df_enquiry['enquiry_dt'],  errors='coerce')
    if 'dt_opened' in df_enquiry.columns:
        df_enquiry['dt_opened']   = pd.to_datetime(df_enquiry['dt_opened'],   errors='coerce')

    snapshot_date = df_account['reporting_dt'].max()
    print(f"  Snapshot date: {snapshot_date}")

  Snapshot date: NaT


# 4.2 Account-Level Features

In [19]:

    # F1: Duration between last payment and account open (per account)
    df_account['diff_lastpaymt_opened_dt'] = (
        df_account['last_paymt_dt'] - df_account['opened_dt']
    ).dt.days

    # F2: Payment history average DPD bucket 0-29
    # paymenthistory1/2 are strings like "000000112000" - parse each char as DPD month
    def mean_dpd_0_29(history_str):
        """Return mean count of months with 0-29 DPD"""
        if pd.isnull(history_str):
            return np.nan
        counts = [1 for ch in str(history_str) if ch in ('0', '1', '2')]
        return np.mean(counts) if counts else 0

    df_account['dpd_0_29_ph1'] = df_account['paymenthistory1'].apply(mean_dpd_0_29)
    df_account['dpd_0_29_ph2'] = df_account['paymenthistory2'].apply(mean_dpd_0_29)
    df_account['payment_history_avg_dpd_0_29'] = df_account[['dpd_0_29_ph1', 'dpd_0_29_ph2']].mean(axis=1)

    # F3: Min months before 30+ DPD first appeared
    def min_months_to_30plus(history_str):
        if pd.isnull(history_str):
            return np.nan
        for i, ch in enumerate(str(history_str)):
            if ch not in ('0', '1', '2', 'X', 'x'):
                return i
        return len(str(history_str))  # never hit 30+

    df_account['min_months_last_30_plus_ph1'] = df_account['paymenthistory1'].apply(min_months_to_30plus)
    df_account['min_months_last_30_plus_ph2'] = df_account['paymenthistory2'].apply(min_months_to_30plus)

In [21]:
# F4: Payment history length
df_account['ph1_length'] = df_account['paymenthistory1'].apply(
    lambda x: len(str(x)) if pd.notnull(x) else np.nan
)
df_account['ph2_length'] = df_account['paymenthistory2'].apply(
    lambda x: len(str(x)) if pd.notnull(x) else np.nan
)

# F5: Utilisation = cur_balance / credit_limit
df_account['utilisation'] = np.where(
    df_account['creditlimit'] > 0,
    df_account['cur_balance_amt'] / df_account['creditlimit'],
    np.nan
)

# 4.3 Aggregate to Customer Level

In [23]:
agg_account = df_account.groupby('customer_no').agg(
    total_diff_lastpaymt_opened_dt   = ('diff_lastpaymt_opened_dt', 'sum'),
    mean_diff_lastpaymt_opened_dt    = ('diff_lastpaymt_opened_dt', 'mean'),
    payment_history_avg_dpd_0_29_bucket = ('payment_history_avg_dpd_0_29', 'mean'),
    min_months_last_30_plus          = ('min_months_last_30_plus_ph1', 'min'),
    payment_history_mean_length      = ('ph1_length', 'mean'),
    Ratio_currbalance_creditlimit    = ('utilisation', 'mean'),
    total_cur_balance                = ('cur_balance_amt', 'sum'),
    total_credit_limit               = ('creditlimit', 'sum'),
    total_cash_limit                 = ('cashlimit', 'sum'),
    mean_credit_limit                = ('creditlimit', 'mean'),
    mean_cash_limit                  = ('cashlimit', 'mean'),
    mean_cur_balance                 = ('cur_balance_amt', 'mean'),
    count_accounts                   = ('acct_type', 'count'),
    amt_past_due_total               = ('amt_past_due', 'sum'),
    mean_rate_of_interest            = ('rateofinterest', 'mean')
).reset_index()

# F6: Utilisation trend
agg_account['utilisation_trend'] = np.where(
    (agg_account['mean_credit_limit'] + agg_account['mean_cash_limit']) > 0,
    (agg_account['total_cur_balance'] / agg_account['total_credit_limit'].replace(0, np.nan)) /
    (agg_account['mean_cur_balance']  / (agg_account['mean_credit_limit'] + agg_account['mean_cash_limit'])),
    np.nan
)

print(f"  Account features aggregated: {agg_account.shape}")

  Account features aggregated: (23896, 17)


# 4.4 Enquiry Features

In [27]:
# Ensure 'enq_amt' is numeric before calculations
df_enquiry['enq_amt'] = pd.to_numeric(df_enquiry['enq_amt'], errors='coerce')

df_enquiry['days_since_enquiry'] = (snapshot_date - df_enquiry['enquiry_dt']).dt.days

agg_enquiry = df_enquiry.groupby('customer_no').agg(
    count_enquiry_recency_365 = ('days_since_enquiry', lambda x: (x <= 365).sum()),
    count_enquiry_recency_90  = ('days_since_enquiry', lambda x: (x <= 90).sum()),
    max_freq_enquiry          = ('enq_purpose', lambda x: x.mode()[0] if len(x) > 0 else np.nan),
    mean_diff_open_enquiry_dt = ('enquiry_dt', lambda x: (
        df_enquiry.loc[x.index, 'enquiry_dt'] -
        df_enquiry.loc[x.index, 'dt_opened']
    ).dt.days.mean()),
    perc_unsecured_others     = ('enq_purpose', lambda x: (x == 'Unsecured').sum() / len(x) if len(x) > 0 else 0),
    total_enquiry_amt         = ('enq_amt', 'sum'),
    mean_enquiry_amt          = ('enq_amt', 'mean'),
    total_enquiries           = ('enq_purpose', 'count'),
).reset_index()

# Encode max_freq_enquiry
le = LabelEncoder()
agg_enquiry['max_freq_enquiry'] = le.fit_transform(
    agg_enquiry['max_freq_enquiry'].astype(str)
)

print(f"  Enquiry features aggregated: {agg_enquiry.shape}")

  Enquiry features aggregated: (23896, 9)


# 4.5 Merge All Tables

In [31]:
# Ensure 'customer_no' columns have consistent data types before merging
# It appears some 'customer_no' columns are int64 and others are object (string).
# Converting them all to int64 for consistency.
df_demographics['customer_no'] = df_demographics['customer_no'].astype(str).str.replace('.0', '', regex=False).astype(int, errors='ignore')
agg_account['customer_no'] = agg_account['customer_no'].astype(str).str.replace('.0', '', regex=False).astype(int, errors='ignore')
agg_enquiry['customer_no'] = agg_enquiry['customer_no'].astype(str).str.replace('.0', '', regex=False).astype(int, errors='ignore')

# Now proceed with merging
df_final = df_demographics.merge(agg_account, on='customer_no', how='left')
df_final = df_final.merge(agg_enquiry,  on='customer_no', how='left')

print(f"  Final merged dataset: {df_final.shape}")

  Final merged dataset: (23896, 107)


# 5: DATA PREPROCESSING

In [32]:
def preprocess(df_final):
    """
    - Drop ID/date columns not useful for modelling
    - Handle missing values (median imputation)
    - Clip extreme outliers at 1st and 99th percentiles
    - Split into X (features) and y (target)
    """
    print("\n" + "="*60)
    print("PREPROCESSING")
    print("="*60)

    drop_cols = ['customer_no', 'dt_opened', 'entry_time', 'upload_dt']
    drop_cols = [c for c in drop_cols if c in df_final.columns]
    df_model = df_final.drop(columns=drop_cols)

    # Separate target
    y = df_model.pop('Bad_label')
    X = df_model.select_dtypes(include=[np.number])

    print(f"  Features before preprocessing : {X.shape[1]}")
    print(f"  Missing values total          : {X.isnull().sum().sum()}")

    # Median imputation
    X = X.fillna(X.median())

    # Clip outliers at 1st/99th percentile
    for col in X.columns:
        low, high = X[col].quantile(0.01), X[col].quantile(0.99)
        X[col] = X[col].clip(low, high)

    print(f"  Class balance: {y.value_counts(normalize=True).to_dict()}")
    print(f"  Features after preprocessing  : {X.shape[1]}")

    return X, y



# 6: HELPER — GINI & RANK ORDERING

In [33]:
def gini_score(y_true, y_prob):
    """Gini = 2 * AUC - 1"""
    auc = roc_auc_score(y_true, y_prob)
    return round((2 * auc - 1) * 100, 4)


def rank_ordering_table(y_true, y_prob, n_deciles=10):
    """
    Splits population into deciles by predicted probability.
    Shows event rate per decile — should be monotonically decreasing
    from Decile 10 (highest risk) to Decile 1 (lowest risk).
    """
    df = pd.DataFrame({'y': y_true.values, 'prob': y_prob})
    df['decile'] = pd.qcut(df['prob'].rank(method='first'), n_deciles, labels=False) + 1
    result = df.groupby('decile').agg(
        Total    = ('y', 'count'),
        Bads     = ('y', 'sum'),
        Bad_Rate = ('y', 'mean')
    ).sort_index(ascending=False)
    result.index.name = 'Decile'
    return result

# 7: MODEL BUILDING

In [34]:
def build_models(X, y):
    """
    Train and evaluate 4 models:
      1. Random Forest
      2. Gradient Boosting
      3. XGBoost
      4. LightGBM  ← typically the strongest for tabular credit data

    Returns best model and train/test splits.
    """
    print("\n" + "="*60)
    print("MODEL TRAINING & EVALUATION")
    print("="*60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    print(f"  Train: {X_train.shape} | Test: {X_test.shape}")

    models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=300, max_depth=6, min_samples_leaf=30,
            class_weight='balanced', n_jobs=-1, random_state=42
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, random_state=42
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
            use_label_encoder=False, eval_metric='auc', random_state=42, n_jobs=-1
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, num_leaves=40,
            class_weight='balanced', random_state=42, n_jobs=-1
        ),
    }

    results = {}
    best_gini = 0
    best_model_name = None
    best_model = None

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_prob_test  = model.predict_proba(X_test)[:, 1]
        y_prob_train = model.predict_proba(X_train)[:, 1]

        gini_train = gini_score(y_train, y_prob_train)
        gini_test  = gini_score(y_test,  y_prob_test)

        results[name] = {
            'model':      model,
            'gini_train': gini_train,
            'gini_test':  gini_test,
            'y_prob':     y_prob_test
        }

        print(f"\n  [{name}]")
        print(f"    Gini Train : {gini_train}")
        print(f"    Gini Test  : {gini_test}  | Benchmark: 37.9")

        if gini_test > best_gini:
            best_gini       = gini_test
            best_model_name = name
            best_model      = model

    print(f"\n  ✓ Best Model: {best_model_name}  |  Gini: {best_gini}")
    return best_model, best_model_name, results, X_train, X_test, y_train, y_test



# 8: MODEL EVALUATION

In [35]:
def evaluate_model(best_model, best_model_name, X_train, X_test, y_train, y_test):
    """
    Full evaluation:
      - Gini score vs benchmark
      - Rank ordering table (decile-wise bad rates)
      - Confusion matrix
      - Feature importance plot
      - SHAP summary plot
    """
    print("\n" + "="*60)
    print(f"EVALUATION: {best_model_name}")
    print("="*60)

    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = best_model.predict(X_test)

    gini = gini_score(y_test, y_prob)
    print(f"\n  Gini Score   : {gini}")
    print(f"  Benchmark    : 37.9")
    print(f"  Improvement  : {round(gini - 37.9, 2)}")

    # ── Classification Report ─────────────────────────────────────
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Good', 'Bad']))

    # ── Rank Ordering ─────────────────────────────────────────────
    print("\n  Rank Ordering Table (Test Set):")
    ro_table = rank_ordering_table(y_test, y_prob)
    print(ro_table.to_string())

    # ── Confusion Matrix Plot ─────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
    plt.title(f"Confusion Matrix — {best_model_name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig("eval_confusion_matrix.png", dpi=150)
    plt.close()

    # ── Feature Importance ────────────────────────────────────────
    if hasattr(best_model, 'feature_importances_'):
        feat_imp = pd.Series(
            best_model.feature_importances_, index=X_test.columns
        ).sort_values(ascending=False).head(20)

        plt.figure(figsize=(9, 6))
        feat_imp.plot(kind='barh', color='steelblue')
        plt.title(f"Top 20 Feature Importances — {best_model_name}")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig("eval_feature_importance.png", dpi=150)
        plt.close()
        print("\n  Top 10 Features:")
        print(feat_imp.head(10).to_string())

    # ── SHAP Summary ──────────────────────────────────────────────
    try:
        explainer = shap.Explainer(best_model, X_test)
        shap_values = explainer(X_test[:500])
        plt.figure()
        shap.summary_plot(shap_values, X_test[:500], plot_type='bar', show=False, max_display=20)
        plt.title("SHAP Feature Importance")
        plt.tight_layout()
        plt.savefig("eval_shap_summary.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("  SHAP plot saved.")
    except Exception as e:
        print(f"  SHAP skipped: {e}")

    return gini, ro_table



# 9: FEATURE MATRIX SUMMARY

In [36]:
FEATURE_MATRIX = pd.DataFrame([
    {"Feature": "payment_history_avg_dpd_0_29_bucket", "Gain": 0.045654, "Description": "Mean count of accounts in 0–29 DPD bucket throughout payment history"},
    {"Feature": "total_diff_lastpaymt_opened_dt",      "Gain": 0.040834, "Description": "Total duration (days) between last payment date and account opened date"},
    {"Feature": "min_months_last_30_plus",             "Gain": 0.038216, "Description": "Smallest number of months before 30+ DPD first appeared"},
    {"Feature": "utilisation_trend",                   "Gain": 0.037524, "Description": "[cur_balance/credit_limit] / [mean_balance/(mean_credit+mean_cash limit)]"},
    {"Feature": "count_enquiry_recency_365",           "Gain": 0.036217, "Description": "Number of enquiries in past 365 days"},
    {"Feature": "Ratio_currbalance_creditlimit",       "Gain": 0.035641, "Description": "Total current balance / total credit limit"},
    {"Feature": "mean_diff_lastpaymt_opened_dt",       "Gain": 0.032186, "Description": "Average duration between last payment date and account open date"},
    {"Feature": "mean_diff_open_enquiry_dt",           "Gain": 0.031058, "Description": "Average difference between enquiry dt_opened and enquiry date"},
    {"Feature": "payment_history_mean_length",         "Gain": 0.029258, "Description": "Average length of payment_history variable"},
    {"Feature": "max_freq_enquiry",                    "Gain": 0.024501, "Description": "Most frequent enquiry purpose"},
    {"Feature": "count_enquiry_recency_90",            "Gain": 0.023797, "Description": "Number of enquiries in past 90 days"},
    {"Feature": "perc_unsecured_others",               "Gain": 0.023428, "Description": "Ratio of unsecured loan enquiries to total enquiries"},
])


def print_feature_matrix():
    print("\n" + "="*60)
    print("FEATURE MATRIX (Top Engineered Features)")
    print("="*60)
    print(FEATURE_MATRIX.to_string(index=False))


# 10: MAIN PIPELINE

In [42]:
def main():
    print("="*60)
    print("Bank GoodCredit — Credit Score Prediction Pipeline")
    print("="*60)

    # STEP 1 — Load Data
    df_account, df_enquiry, df_demographics = load_data()

    # STEP 2 — EDA
    perform_eda(df_account, df_enquiry, df_demographics)

    # STEP 3 — Feature Engineering
    df_final = engineer_features(df_account, df_enquiry, df_demographics)

    # STEP 4 — Preprocessing
    X, y = preprocess(df_final)

    # STEP 5 — Feature Matrix (from brief)
    print_feature_matrix()

    # STEP 6 — Model Building
    best_model, best_model_name, results, X_train, X_test, y_train, y_test = build_models(X, y)

    # STEP 7 — Evaluation
    final_gini, rank_table = evaluate_model(
        best_model, best_model_name, X_train, X_test, y_train, y_test
    )

    # STEP 8 — Summary
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    print(f"  Best Model      : {best_model_name}")
    print(f"  Gini (Test)     : {final_gini}")
    print(f"  Benchmark Gini  : 37.9")
    print(f"  Beat Benchmark? : {'YES ✓' if final_gini > 37.9 else 'NO — needs tuning'}")
    print("="*60)


if __name__ == "__main__":
    main()



Bank GoodCredit — Credit Score Prediction Pipeline
Loading tables from database...
  Cust_Account     : (186329, 21)
  Cust_Enquiry     : (413188, 6)
  Cust_Demographics: (23896, 83)

EXPLORATORY DATA ANALYSIS

FEATURE ENGINEERING
  Snapshot date: 2015-12-20 00:00:00
  Account features aggregated: (23896, 17)
  Enquiry features aggregated: (23896, 9)
  Final merged dataset: (23896, 107)

PREPROCESSING
  Features before preprocessing : 24
  Missing values total          : 30265
  Class balance: {'0': 0.9579845999330432, '1': 0.04201540006695681}
  Features after preprocessing  : 24

FEATURE MATRIX (Top Engineered Features)
                            Feature     Gain                                                               Description
payment_history_avg_dpd_0_29_bucket 0.045654      Mean count of accounts in 0–29 DPD bucket throughout payment history
     total_diff_lastpaymt_opened_dt 0.040834   Total duration (days) between last payment date and account opened date
            m

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['0' '1']